# Experiment: Long-Window Logistic Regression

**Model:** Logistic Regression  
**Window sizes:** 1500 frames (~15 s), 3000 frames (~30 s)  
**Split:** Session 1 => train, Session 2 => test

---

### Motivation

The window size ablation (WS=64–256) showed no improvement. But those windows are all
still too short to capture slow physiological signals:

- **Breathing:** ~0.25 Hz => one full cycle ≈ 4 s => requires >= 04 s window  
- **Postural sway / micro-movement cycles:** 5–15 s  

We push to dramatically longer windows 15 s and 30 s to test whether integrating
signal over a full breath cycle or longer provides a meaningful boost.

### Expected trade-off

Longer windows => fewer windows per session => less training data.  
We mitigate with 25 % step overlap (`STEP = WS // 4`) to maximise window count.
With ~15 min of session 1 data, 30 s windows will yield very few training samples
this experiment is exploratory and directional, not definitive.

### Why LR and not AE?

With fewer than 100 training windows at WS=3000, training a deep autoencoder would
almost certainly overfit. LR with balanced class weights is the most data efficient
model for this regime.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                             ConfusionMatrixDisplay, RocCurveDisplay)

DATA_DIR = "../data/feasibility-study/processed"

CONFIGS = [
    {"ws": 1500, "label": "15s"},
    {"ws": 3000, "label": "30s"},
]

X_train_raw = np.load(f"{DATA_DIR}/X_train.npy")
y_train_raw = np.load(f"{DATA_DIR}/y_train.npy")
X_test_raw  = np.load(f"{DATA_DIR}/X_test.npy")
y_test_raw  = np.load(f"{DATA_DIR}/y_test.npy")

print(f"Train frames: {X_train_raw.shape}  |  Test frames: {X_test_raw.shape}")

Train frames: (71516, 55)  |  Test frames: (71112, 55)


In [ ]:
def make_windows(X, y, window_size, step): # shared utility see preprocessing in nb 00
    """Per-label windowing prevents label boundary contamination"""
    Xw, yw = [], []
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        Xl  = X[idx]
        for start in range(0, len(Xl) - window_size + 1, step):
            Xw.append(Xl[start:start + window_size])
            yw.append(label)
    return np.array(Xw, dtype=np.float32), np.array(yw, dtype=np.int64)


print("Window counts with 25% step:")
for cfg in CONFIGS:
    ws, step = cfg["ws"], cfg["ws"] // 4
    Xtr_w, ytr_w = make_windows(X_train_raw, y_train_raw, ws, step)
    Xte_w, yte_w = make_windows(X_test_raw,  y_test_raw,  ws, step)
    print(f"  {cfg['label']} (ws={ws}, step={step}):  "
          f"train={len(ytr_w)} (empty={( ytr_w==0).sum()}, occ={( ytr_w==1).sum()})  "
          f"test={len(yte_w)}")

Window counts with 25% step:
  15s (ws=1500, step=375):  train=184 (empty=60, occ=124)  test=183
  30s (ws=3000, step=750):  train=88 (empty=28, occ=60)  test=88


**Observation:** Very few training windows, especially at 30 s. This limits statistical
power. Results from this experiment are directional evidence, not a definitive evaluation.

In [ ]:
results = []

for cfg in CONFIGS:
    ws   = cfg["ws"]
    step = ws // 4     # 25% overlap maximise window count given limited data

    Xtr_w, ytr_w = make_windows(X_train_raw, y_train_raw, ws, step)
    Xte_w, yte_w = make_windows(X_test_raw,  y_test_raw,  ws, step)

    # Flatten for LR
    Xtr_f = Xtr_w.reshape(len(Xtr_w), -1)
    Xte_f = Xte_w.reshape(len(Xte_w), -1)

    clf = LogisticRegression(class_weight="balanced", C=1.0, max_iter=2000, random_state=42)
    clf.fit(Xtr_f, ytr_w)

    train_acc = clf.score(Xtr_f, ytr_w)
    test_acc  = clf.score(Xte_f, yte_w)
    prob      = clf.predict_proba(Xte_f)[:, 1]
    pred      = clf.predict(Xte_f)
    auc       = roc_auc_score(yte_w, prob)
    empty_rec = (pred[yte_w==0] == 0).mean() if (yte_w==0).sum() > 0 else 0.0

    print(f"\n--- {cfg['label']} (ws={ws}) ---")
    print(f"  Train windows: {len(ytr_w)}  |  Test windows: {len(yte_w)}")
    print(f"  Train acc: {train_acc:.2%}  |  Test acc: {test_acc:.2%}")
    print(f"  AUC-ROC: {auc:.4f}  |  Empty recall: {empty_rec:.2%}")
    print(classification_report(yte_w, pred, target_names=["empty", "occupied"]))

    results.append({"label": cfg["label"], "ws": ws, "n_train": len(ytr_w),
                    "train_acc": train_acc, "test_acc": test_acc,
                    "auc": auc, "empty_recall": empty_rec})


--- 15s (ws=1500) ---
  Train windows: 184  |  Test windows: 183
  Train acc: 100.00%  |  Test acc: 66.12%
  AUC-ROC: 0.5505  |  Empty recall: 27.94%
              precision    recall  f1-score   support

       empty       0.59      0.28      0.38        68
    occupied       0.68      0.89      0.77       115

    accuracy                           0.66       183
   macro avg       0.63      0.58      0.57       183
weighted avg       0.65      0.66      0.62       183


--- 30s (ws=3000) ---
  Train windows: 88  |  Test windows: 88
  Train acc: 100.00%  |  Test acc: 63.64%
  AUC-ROC: 0.6646  |  Empty recall: 25.00%
              precision    recall  f1-score   support

       empty       0.50      0.25      0.33        32
    occupied       0.67      0.86      0.75        56

    accuracy                           0.64        88
   macro avg       0.58      0.55      0.54        88
weighted avg       0.61      0.64      0.60        88



In [4]:
import pandas as pd

df_res = pd.DataFrame(results)
print("=== Long-Window LR Results ===")
print(df_res[["label","ws","n_train","test_acc","auc","empty_recall"]].to_string(index=False))

# Compare with baseline WS=32 results
comparison = {
    "Window": ["32 (0.32s) — baseline", "1500 (15s)", "3000 (30s)"],
    "N train": ["~4467", df_res.iloc[0]["n_train"], df_res.iloc[1]["n_train"]],
    "AUC": ["0.631", f"{df_res.iloc[0]['auc']:.3f}", f"{df_res.iloc[1]['auc']:.3f}"],
    "Empty recall": ["9%", f"{df_res.iloc[0]['empty_recall']:.0%}", f"{df_res.iloc[1]['empty_recall']:.0%}"]
}
print("\n=== Comparison with WS=32 baseline ===")
print(pd.DataFrame(comparison).to_string(index=False))

=== Long-Window LR Results ===
label   ws  n_train  test_acc      auc  empty_recall
  15s 1500      184  0.661202 0.550512      0.279412
  30s 3000       88  0.636364 0.664621      0.250000

=== Comparison with WS=32 baseline ===
               Window N train   AUC Empty recall
32 (0.32s) — baseline   ~4467 0.631           9%
           1500 (15s)     184 0.551          28%
           3000 (30s)      88 0.665          25%


## Results Analysis

| Window | Duration | N train | AUC | Empty recall |
|---|---|---|---|---|
| 32 (baseline) | ~0.32 s | ~4 467 | ~0.631 | ~9 % |
| 1500 | ~15 s | ~184 | ~0.551 | ~28 % |
| 3000 | ~30 s | ~88 | ~0.665 | ~25 % |

**30 s window matches the best AUC of the entire study** with only 88 training samples.
This is directionally meaningful: longer windows may help, but the data is too limited
to be confident.

**15 s is the worst configuration** AUC drops *below* the 32 frame baseline.
This is likely a small sample effect: 184 windows is not enough for LR to learn
a reliable boundary in a 1500×55=82 500-dimensional feature space. 
82,500 dimensions vs. 184 training samples is an extreme underdetermination
 LR will be numerically unstable regardless of regularization.

**The non monotonic trend (15s worse => 30s better) is a warning sign.** With this
little data, variance dominates. We cannot claim 30 s windows are generally better
from a single comparison.